<a href="https://colab.research.google.com/github/sanchitjobs13-dotcom/machinelearningtest1/blob/dev/ml22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler
import matplotlib.pyplot as plt

In [ ]:
broker1_df = pd.read_csv("./sample_data/Broker1.csv")
broker2_df = pd.read_csv("./sample_data/Broker2.csv")
broker3_df = pd.read_csv("./sample_data/Broker3.csv")
prod_master_df = pd.read_csv("./sample_data/product_master.csv")
reg_master_df = pd.read_csv("./sample_data/region_master.csv")

Map the Brokers to a Common DataSet

In [ ]:

# 2. Unnest 'aliases' into lookup tables for string matching
prod_expanded = prod_master_df.assign(match_key=prod_master_df['aliases'].str.split('|')).explode('match_key')
prod_expanded['match_key'] = prod_expanded['match_key'].str.strip().str.lower()
prod_std = prod_master_df.copy().assign(match_key=prod_master_df['product_name_standardized'].str.strip().str.lower())
prod_lookup = pd.concat([prod_expanded, prod_std]).drop_duplicates(subset=['match_key'])[
    ['match_key', 'product_id', 'product_name_standardized', 'product_grade_standard', 'default_unit_of_measure']
]

reg_expanded = reg_master_df.assign(match_key=reg_master_df['aliases'].str.split('|')).explode('match_key')
reg_expanded['match_key'] = reg_expanded['match_key'].str.strip().str.lower()
reg_std = reg_master_df.copy().assign(match_key=reg_master_df['market_region_standardized'].str.strip().str.lower())
reg_lookup = pd.concat([reg_expanded, reg_std]).drop_duplicates(subset=['match_key'])[
    ['match_key', 'market_region_standardized']
]

# 3. Standardize broker schemas to raw common format
b1_mapped = pd.DataFrame({
    'source_name': 'Broker1',
    'source_file_name': 'Broker1.csv',
    'source_row_number': broker1_df.index + 2,
    'product_name_raw': broker1_df['Commodity'],
    'product_grade': broker1_df['Grade'],
    'raw_region': broker1_df['Region'],
    'pricing_date': broker1_df['Date'],
    'trade_date': broker1_df['Date'],
    'price': broker1_df['Last_Trade'],
    'currency': 'USD',
    'quantity': broker1_df['Volume'],
    'broker': 'Broker1',
    'counterparty': broker1_df['Trader']
})

b2_mapped = pd.DataFrame({
    'source_name': 'Broker2',
    'source_file_name': 'Broker2.csv',
    'source_row_number': broker2_df.index + 2,
    'product_name_raw': broker2_df['Product_Name'],
    'product_grade': broker2_df['Grade_Spec'],
    'raw_region': broker2_df['Market'],
    'pricing_date': broker2_df['Pricing_Date'],
    'trade_date': broker2_df['Pricing_Date'],
    'price': broker2_df['Price_USD_per_Unit'],
    'currency': 'USD',
    'unit_of_measure': broker2_df['UOM'],
    'quantity': np.nan,
    'broker': 'Broker2',
    'counterparty': broker2_df['Counterparty']
})

b3_mapped = pd.DataFrame({
    'source_name': 'Broker3',
    'source_file_name': 'Broker3.csv',
    'source_row_number': broker3_df.index + 2,
    'product_name_raw': broker3_df['Product_Description'],
    'product_grade': np.nan,
    'raw_region': broker3_df['Location'],
    'pricing_date': broker3_df['Timestamp'].astype(str).str.split().str[0],
    'trade_date': broker3_df['Timestamp'].astype(str).str.split().str[0],
    'price': broker3_df['Trading_Price'],
    'currency': broker3_df['Currency'],
    'unit_of_measure': broker3_df['Unit'],
    'quantity': broker3_df['Quantity_Traded'],
    'broker': broker3_df['Broker_Name'],
    'counterparty': np.nan
})

# 4. Merge datasets and map against lookup tables
combined_df = pd.concat([b1_mapped, b2_mapped, b3_mapped], ignore_index=True)

combined_df['prod_key'] = combined_df['product_name_raw'].astype(str).str.strip().str.lower()
combined_df['reg_key'] = combined_df['raw_region'].astype(str).str.strip().str.lower()

merged_df = combined_df.merge(prod_lookup, left_on='prod_key', right_on='match_key', how='left')
merged_df = merged_df.merge(reg_lookup, left_on='reg_key', right_on='match_key', how='left')

# 5. Fill defaults and validation metadata
merged_df['unit_of_measure'] = merged_df['unit_of_measure'].fillna(merged_df['default_unit_of_measure'])
merged_df['product_grade'] = merged_df['product_grade'].fillna(merged_df['product_grade_standard'])
merged_df['market_region'] = merged_df['market_region_standardized']
merged_df['source_id'] = [f"SRC-{i+1:04d}" for i in range(len(merged_df))]
merged_df['source_sheet_name'] = 'Sheet1'
merged_df['observation_type'] = 'Trade'
merged_df['confidence_score'] = np.where(merged_df['product_id'].notna() & merged_df['market_region'].notna(), 1.0, 0.7)
merged_df['validation_status'] = np.where(merged_df['confidence_score'] == 1.0, 'Validated', 'Pending')
merged_df['review_required'] = merged_df['confidence_score'] < 1.0
merged_df['created_timestamp'] = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')

# 6. Reorder to final target schema and export
final_columns = [
    'source_id', 'source_name', 'source_file_name', 'source_sheet_name', 'source_row_number',
    'product_name_raw', 'product_id', 'product_name_standardized', 'product_grade',
    'market_region', 'pricing_date', 'trade_date', 'price', 'currency', 'unit_of_measure',
    'quantity', 'observation_type', 'broker', 'counterparty', 'confidence_score',
    'validation_status', 'review_required', 'created_timestamp'
]

raw_common_extracted_records = merged_df[final_columns]
raw_common_extracted_records.to_csv("./sample_data/Raw_common_extracted_records.csv", index=False)

In [ ]:
raw_common_extracted_records.head()

,source_id,source_name,source_file_name,source_sheet_name,source_row_number,product_name_raw,product_id,product_name_standardized,product_grade,market_region,...,currency,unit_of_measure,quantity,observation_type,broker,counterparty,confidence_score,validation_status,review_required,created_timestamp
0,SRC-0001,Broker1,Broker1.csv,Sheet1,2,WTI Crude,PROD-WTI-001,WTI Crude Oil,Light Sweet,US Gulf Coast,...,USD,BBL,500.0,Trade,Broker1,Internal Trader,1.0,Validated,False,2026-08-19 11:05:11
1,SRC-0002,Broker1,Broker1.csv,Sheet1,3,ULSD,PROD-ULSD-001,Ultra-Low-Sulfur Diesel,15 ppm,US Gulf Coast,...,USD,GAL,1000.0,Trade,Broker1,John Smith,1.0,Validated,False,2026-08-19 11:05:11
2,SRC-0003,Broker1,Broker1.csv,Sheet1,4,Brent Crude,PROD-BRENT-001,Brent Crude Oil,Light Sweet,North Sea,...,USD,BBL,750.0,Trade,Broker1,EU Trader,1.0,Validated,False,2026-08-19 11:05:11
3,SRC-0004,Broker1,Broker1.csv,Sheet1,5,Premium Gas,PROD-PREMGAS-001,Premium Gasoline,92 Octane,US Gulf Coast,...,USD,GAL,500.0,Trade,Broker1,Jane Doe,1.0,Validated,False,2026-08-19 11:05:11
4,SRC-0005,Broker1,Broker1.csv,Sheet1,6,Fuel Oil No 2,PROD-FUELOIL-001,Fuel Oil No. 2,Standard,US Gulf Coast,...,USD,GAL,250.0,Trade,Broker1,Bob Johnson,1.0,Validated,False,2026-08-19 11:05:11


In [ ]:
raw_common_extracted_records.describe()

,source_row_number,price,quantity,confidence_score
count,29.000000,29.000000,20.00000,29.000000
mean,6.344828,33.662724,620.00000,0.979310
std,2.844492,37.696380,259.24993,0.077364
min,2.000000,2.410000,250.00000,0.700000
25%,4.000000,2.500000,500.00000,1.000000
50%,6.000000,2.680000,600.00000,1.000000
75%,9.000000,77.050000,762.50000,1.000000
max,11.000000,78.600000,1200.00000,1.000000


In [ ]:
raw_common_extracted_records.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29 entries, 0 to 28
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   source_id                  29 non-null     object 
 1   source_name                29 non-null     object 
 2   source_file_name           29 non-null     object 
 3   source_sheet_name          29 non-null     object 
 4   source_row_number          29 non-null     int64  
 5   product_name_raw           29 non-null     object 
 6   product_id                 27 non-null     object 
 7   product_name_standardized  27 non-null     object 
 8   product_grade              27 non-null     object 
 9   market_region              29 non-null     object 
 10  pricing_date               29 non-null     object 
 11  trade_date                 29 non-null     object 
 12  price                      29 non-null     float64
 13  currency                   29 non-null     object 
 

Data Cleaning and Imputation

In [ ]:
# 1. Read Raw Common Extracted File
common_rec_df = pd.read_csv("./sample_data/Raw_common_extracted_records.csv")

# 2. Impute Missing Product Master Lookups (Unmapped Raw Names)
product_fallback = {
    'reformulated gasoline': {
        'product_id': 'PROD-REGGAS-001',
        'product_name_standardized': 'Regular Gasoline',
        'product_grade': '87 Octane',
        'unit_of_measure': 'GAL'
    }
}

for idx, row in common_rec_df.iterrows():
    if pd.isna(row['product_id']):
        raw_name = str(row['product_name_raw']).strip().lower()
        if raw_name in product_fallback:
            match = product_fallback[raw_name]
            common_rec_df.at[idx, 'product_id'] = match['product_id']
            common_rec_df.at[idx, 'product_name_standardized'] = match['product_name_standardized']
            common_rec_df.at[idx, 'product_grade'] = match['product_grade']
            common_rec_df.at[idx, 'unit_of_measure'] = match['unit_of_measure']

# 3. Impute Missing Quantities (Group-level Median with Global Fallback)
common_rec_df['quantity'] = common_rec_df.groupby('product_id')['quantity'].transform(lambda x: x.fillna(x.median()))
common_rec_df['quantity'] = common_rec_df['quantity'].fillna(common_rec_df['quantity'].median())

# 4. Impute Missing Categorical Attributes
common_rec_df['counterparty'] = common_rec_df['counterparty'].fillna('Unknown / Benchmark')

# 5. Clean Date Formats & Re-evaluate Data Quality Flags
common_rec_df['pricing_date'] = pd.to_datetime(common_rec_df['pricing_date']).dt.strftime('%Y-%m-%d')
common_rec_df['trade_date'] = pd.to_datetime(common_rec_df['trade_date']).dt.strftime('%Y-%m-%d')

common_rec_df['confidence_score'] = np.where(
    common_rec_df['product_id'].notna() & common_rec_df['market_region'].notna(),
    1.0,
    0.7
)
common_rec_df['validation_status'] = np.where(common_rec_df['confidence_score'] == 1.0, 'Validated', 'Pending')
common_rec_df['review_required'] = common_rec_df['confidence_score'] < 1.0
common_rec_df['created_timestamp'] = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')

# 6. Save Cleaned & Imputed Dataset
common_rec_df.to_csv("./sample_data/Cleaned_common_extracted_records.csv", index=False)

In [ ]:
common_rec_df.head()

,source_id,source_name,source_file_name,source_sheet_name,source_row_number,product_name_raw,product_id,product_name_standardized,product_grade,market_region,...,currency,unit_of_measure,quantity,observation_type,broker,counterparty,confidence_score,validation_status,review_required,created_timestamp
0,SRC-0001,Broker1,Broker1.csv,Sheet1,2,WTI Crude,PROD-WTI-001,WTI Crude Oil,Light Sweet,US Gulf Coast,...,USD,BBL,500.0,Trade,Broker1,Internal Trader,1.0,Validated,False,2026-08-19 11:05:11
1,SRC-0002,Broker1,Broker1.csv,Sheet1,3,ULSD,PROD-ULSD-001,Ultra-Low-Sulfur Diesel,15 ppm,US Gulf Coast,...,USD,GAL,1000.0,Trade,Broker1,John Smith,1.0,Validated,False,2026-08-19 11:05:11
2,SRC-0003,Broker1,Broker1.csv,Sheet1,4,Brent Crude,PROD-BRENT-001,Brent Crude Oil,Light Sweet,North Sea,...,USD,BBL,750.0,Trade,Broker1,EU Trader,1.0,Validated,False,2026-08-19 11:05:11
3,SRC-0004,Broker1,Broker1.csv,Sheet1,5,Premium Gas,PROD-PREMGAS-001,Premium Gasoline,92 Octane,US Gulf Coast,...,USD,GAL,500.0,Trade,Broker1,Jane Doe,1.0,Validated,False,2026-08-19 11:05:11
4,SRC-0005,Broker1,Broker1.csv,Sheet1,6,Fuel Oil No 2,PROD-FUELOIL-001,Fuel Oil No. 2,Standard,US Gulf Coast,...,USD,GAL,250.0,Trade,Broker1,Bob Johnson,1.0,Validated,False,2026-08-19 11:05:11


Feature Engineering

In [ ]:

# 1. Load Cleaned Dataset
common_rec_df = pd.read_csv("./sample_data/Cleaned_common_extracted_records.csv")

# 2. Extract Temporal Features
common_rec_df['pricing_date'] = pd.to_datetime(common_rec_df['pricing_date'])
common_rec_df['trade_date'] = pd.to_datetime(common_rec_df['trade_date'])

common_rec_df['pricing_year'] = common_rec_df['pricing_date'].dt.year
common_rec_df['pricing_month'] = common_rec_df['pricing_date'].dt.month
common_rec_df['pricing_day'] = common_rec_df['pricing_date'].dt.day
common_rec_df['pricing_dayofweek'] = common_rec_df['pricing_date'].dt.dayofweek
common_rec_df['is_weekend'] = common_rec_df['pricing_dayofweek'].isin([5, 6]).astype(int)

# 3. Derive Monetary & Notional Metrics
common_rec_df['total_notional_value'] = common_rec_df['price'] * common_rec_df['quantity']

# 4. Standardize Pricing & Volume to Barrels (1 BBL = 42 Gallons)
common_rec_df['price_per_bbl'] = np.where(
    common_rec_df['unit_of_measure'] == 'GAL',
    common_rec_df['price'] * 42.0,
    common_rec_df['price']
)

common_rec_df['quantity_bbl'] = np.where(
    common_rec_df['unit_of_measure'] == 'GAL',
    common_rec_df['quantity'] / 42.0,
    common_rec_df['quantity']
)

# 5. Calculate Market Benchmark Deviations
product_avg_price = common_rec_df.groupby('product_id')['price_per_bbl'].transform('mean')
common_rec_df['price_pct_diff_from_avg'] = (
    (common_rec_df['price_per_bbl'] - product_avg_price) / product_avg_price
) * 100

# 6. One-Hot Encode Categorical Features
categorical_cols = ['market_region', 'observation_type', 'currency']
common_rec_df = pd.get_dummies(common_rec_df, columns=categorical_cols, prefix=categorical_cols, drop_first=False)

# 7. Save Feature-Engineered Dataset
common_rec_df.to_csv("./sample_data/Feature_engineered_records.csv", index=False)

In [ ]:
common_rec_df.head()

,source_id,source_name,source_file_name,source_sheet_name,source_row_number,product_name_raw,product_id,product_name_standardized,product_grade,pricing_date,...,is_weekend,total_notional_value,price_per_bbl,quantity_bbl,price_pct_diff_from_avg,market_region_North Sea,market_region_US Gulf Coast,market_region_US Northeast,observation_type_Trade,currency_USD
0,SRC-0001,Broker1,Broker1.csv,Sheet1,2,WTI Crude,PROD-WTI-001,WTI Crude Oil,Light Sweet,2026-07-15,...,0,39225.00,78.450,500.000000,-0.095511,False,True,False,True,True
1,SRC-0002,Broker1,Broker1.csv,Sheet1,3,ULSD,PROD-ULSD-001,Ultra-Low-Sulfur Diesel,15 ppm,2026-07-15,...,0,2415.00,101.430,23.809524,48.159509,False,True,False,True,True
2,SRC-0003,Broker1,Broker1.csv,Sheet1,4,Brent Crude,PROD-BRENT-001,Brent Crude Oil,Light Sweet,2026-07-15,...,0,57675.00,76.900,750.000000,-0.086618,True,False,False,True,True
3,SRC-0004,Broker1,Broker1.csv,Sheet1,5,Premium Gas,PROD-PREMGAS-001,Premium Gasoline,92 Octane,2026-07-15,...,0,1294.50,108.738,11.904762,-0.211987,False,True,False,True,True
4,SRC-0005,Broker1,Broker1.csv,Sheet1,6,Fuel Oil No 2,PROD-FUELOIL-001,Fuel Oil No. 2,Standard,2026-07-15,...,0,668.75,112.350,5.952381,-0.279590,False,True,False,True,True


Data Encoding

In [ ]:
# 1. Load Feature Engineered Dataset
common_rec_df = pd.read_csv("./sample_data/Feature_engineered_records.csv")

# 2. Label Encoding (for Ordinal / Key Entity Identifiers)
label_enc_product = LabelEncoder()
common_rec_df['product_id_encoded'] = label_enc_product.fit_transform(common_rec_df['product_id'])

label_enc_grade = LabelEncoder()
common_rec_df['product_grade_encoded'] = label_enc_grade.fit_transform(common_rec_df['product_grade'])

# 3. One-Hot Encoding (for Nominal Categorical Variables)
one_hot_columns = ['source_name', 'unit_of_measure', 'broker']
common_rec_df = pd.get_dummies(
    common_rec_df,
    columns=one_hot_columns,
    prefix=one_hot_columns,
    drop_first=False,
    dtype=int
)

# 4. Frequency Encoding (for High-Cardinality Categorical Features)
counterparty_freq = common_rec_df['counterparty'].value_counts(normalize=True).to_dict()
common_rec_df['counterparty_freq_encoded'] = common_rec_df['counterparty'].map(counterparty_freq)

# 5. Binary Encoding (for Boolean Flags)
boolean_columns = ['review_required', 'is_weekend']
for col in boolean_columns:
    if col in common_rec_df.columns:
        common_rec_df[col] = common_rec_df[col].astype(int)

# Convert any remaining dummy boolean columns to 0/1 integers
bool_cols = common_rec_df.select_dtypes(include=['bool']).columns
common_rec_df[bool_cols] = common_rec_df[bool_cols].astype(int)

# 6. Save Encoded Dataset
common_rec_df.to_csv("./sample_data/Encoded_common_records.csv", index=False)

In [ ]:
common_rec_df.head()

,source_id,source_file_name,source_sheet_name,source_row_number,product_name_raw,product_id,product_name_standardized,product_grade,pricing_date,trade_date,...,unit_of_measure_GAL,unit_of_measure_Gallon,broker_Broker1,broker_Broker2,broker_Commerical Bank A,broker_Energy Trader B,broker_International Broker C,broker_Regional Distributor D,broker_Supplier E,counterparty_freq_encoded
0,SRC-0001,Broker1.csv,Sheet1,2,WTI Crude,PROD-WTI-001,WTI Crude Oil,Light Sweet,2026-07-15,2026-07-15,...,0,0,1,0,0,0,0,0,0,0.068966
1,SRC-0002,Broker1.csv,Sheet1,3,ULSD,PROD-ULSD-001,Ultra-Low-Sulfur Diesel,15 ppm,2026-07-15,2026-07-15,...,1,0,1,0,0,0,0,0,0,0.068966
2,SRC-0003,Broker1.csv,Sheet1,4,Brent Crude,PROD-BRENT-001,Brent Crude Oil,Light Sweet,2026-07-15,2026-07-15,...,0,0,1,0,0,0,0,0,0,0.068966
3,SRC-0004,Broker1.csv,Sheet1,5,Premium Gas,PROD-PREMGAS-001,Premium Gasoline,92 Octane,2026-07-15,2026-07-15,...,1,0,1,0,0,0,0,0,0,0.068966
4,SRC-0005,Broker1.csv,Sheet1,6,Fuel Oil No 2,PROD-FUELOIL-001,Fuel Oil No. 2,Standard,2026-07-15,2026-07-15,...,1,0,1,0,0,0,0,0,0,0.068966


Fully Encoded

In [ ]:

# 1. Load the partially encoded dataset
complete_enc_df = pd.read_csv('./sample_data/Encoded_common_records.csv')

# --- Safety Step: Check columns to prevent KeyError ---
current_columns = set(complete_enc_df.columns)

# 2. Date Decomposition Encoding (safe)
date_cols_expected = ['pricing_date', 'trade_date', 'created_timestamp']
date_cols_present = [col for col in date_cols_expected if col in current_columns]

for col in date_cols_present:
    complete_enc_df[col] = pd.to_datetime(complete_enc_df[col], errors='coerce')
    # Safe prefix that avoids existing standard column name conflicts
    prefix = col.replace('_timestamp', '').replace('_date', '').replace(' ', '_')

    complete_enc_df[f'{prefix}_year_fe'] = complete_enc_df[col].dt.year
    complete_enc_df[f'{prefix}_month_fe'] = complete_enc_df[col].dt.month
    complete_enc_df[f'{prefix}_day_fe'] = complete_enc_df[col].dt.day
    complete_enc_df[f'{prefix}_dayofweek_fe'] = complete_enc_df[col].dt.dayofweek
    # Drop original datetime columns safely
    complete_enc_df = complete_enc_df.drop(columns=[col])

# 3. Label Encoding (For entity identifiers and statuses - safe)
label_encode_cols_expected = ['product_id', 'product_grade', 'counterparty', 'validation_status']
label_encode_cols_present = [col for col in label_encode_cols_expected if col in current_columns]

for col in label_encode_cols_present:
    # Fill missing values explicitly before encoding
    complete_enc_df[col] = complete_enc_df[col].fillna("MISSING")

    le = LabelEncoder()
    # Appending safe suffix
    complete_enc_df[f'{col}_le_encoded'] = le.fit_transform(complete_enc_df[col].astype(str))
    # Drop original categorical columns safely
    complete_enc_df = complete_enc_df.drop(columns=[col])

# 4. Drop High-Cardinality/Text Fields not suitable for standard encoding (safe)
cols_to_drop_expected = ['source_id', 'product_name_raw', 'product_name_standardized',
                          'source_file_name', 'source_sheet_name']
cols_to_drop_present = [col for col in cols_to_drop_expected if col in current_columns]

complete_enc_df = complete_enc_df.drop(columns=cols_to_drop_present)

# 5. One-Hot Encoding Finalization (Ensure all candidates are 0/1 integers)
# This handles boolean types and strings that look like booleans
bool_candidates = complete_enc_df.select_dtypes(include=['bool', 'object']).columns
for col in bool_candidates:
    non_nulls = complete_enc_df[col].dropna().unique()
    # Check if unique non-null values consist only of boolean-like items
    if set(non_nulls) <= {True, False, 1, 0, 'True', 'False', '1', '0', 'true', 'false'}:
        complete_enc_df[col] = complete_enc_df[col].astype(bool).astype(int)

# 6. Final Review to guarantee complete numeric encoding
print("=== Complete Encoding Status Summary ===")
print("Final Data Types Summary:\n")
print(complete_enc_df.dtypes.value_counts())

# Save Fully Encoded Dataset
complete_enc_df.to_csv("./sample_data/Fully_Encoded_Common_Records_Safety.csv", index=False)
print("\nSaved fully numeric dataset to: Fully_Encoded_Common_Records_Safety.csv")

=== Complete Encoding Status Summary ===
Final Data Types Summary:

int64      32
int32      12
float64     8
Name: count, dtype: int64

Saved fully numeric dataset to: Fully_Encoded_Common_Records_Safety.csv


In [ ]:
complete_enc_df.head()

,source_row_number,price,quantity,confidence_score,review_required,pricing_year,pricing_month,pricing_day,pricing_dayofweek,is_weekend,...,trade_day_fe,trade_dayofweek_fe,created_year_fe,created_month_fe,created_day_fe,created_dayofweek_fe,product_id_le_encoded,product_grade_le_encoded,counterparty_le_encoded,validation_status_le_encoded
0,2,78.450,500.0,1.0,0,2026,7,15,2,0,...,15,2,2026,8,19,2,6,5,3,0
1,3,2.415,1000.0,1.0,0,2026,7,15,2,0,...,15,2,2026,8,19,2,5,1,5,0
2,4,76.900,750.0,1.0,0,2026,7,15,2,0,...,15,2,2026,8,19,2,0,5,1,0
3,5,2.589,500.0,1.0,0,2026,7,15,2,0,...,15,2,2026,8,19,2,3,4,4,0
4,6,2.675,250.0,1.0,0,2026,7,15,2,0,...,15,2,2026,8,19,2,1,7,0,0


Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

# Load the fully encoded dataset
df = pd.read_csv("./sample_data/Fully_Encoded_Common_Records_Safety.csv")

# Drop any non-feature columns (like source_row_number) if present
df_scaled = df.drop(columns=['source_row_number'], errors='ignore')

# Optionally, you may want to keep the target (price) unscaled? If so, separate it.
# But we will scale everything for simplicity. If you want to keep price unscaled,
# you can exclude it from scaling and then recombine.

# Identify columns to scale: all numeric columns (they are all numeric)
# We'll scale all columns except maybe 'price' if you want to keep it unscaled.
# The user said "I am thinking to predict price" so they might want to keep price unscaled.
# We'll provide option to exclude target from scaling.

# Choose to exclude 'price' from scaling? Set to True if you want price unscaled.
exclude_target = True  # change to False if you want to scale price too

if exclude_target and 'price' in df_scaled.columns:
    # Separate target and features
    features = df_scaled.drop(columns=['price'])
    target = df_scaled[['price']]  # keep as DataFrame
    # Scale features
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    features_scaled_df = pd.DataFrame(features_scaled, columns=features.columns)
    # Recombine with target (unscaled)
    df_scaled_final = pd.concat([features_scaled_df, target.reset_index(drop=True)], axis=1)
else:
    # Scale everything
    scaler = StandardScaler()
    df_scaled_values = scaler.fit_transform(df_scaled)
    df_scaled_final = pd.DataFrame(df_scaled_values, columns=df_scaled.columns)

# Save scaled dataset
df_scaled_final.to_csv("./sample_data/Scaled_Common_Records.csv", index=False)

print("Scaling complete. Scaled dataset saved.")

Scaling complete. Scaled dataset saved.


In [ ]:
df_scaled_final.head()

,quantity,confidence_score,review_required,pricing_year,pricing_month,pricing_day,pricing_dayofweek,is_weekend,total_notional_value,price_per_bbl,...,trade_dayofweek_fe,created_year_fe,created_month_fe,created_day_fe,created_dayofweek_fe,product_id_le_encoded,product_grade_le_encoded,counterparty_le_encoded,validation_status_le_encoded,price
0,-0.554836,0.0,0.0,0.0,0.0,-1.035098,-1.035098,0.0,0.746179,-0.057685,...,-1.035098,0.0,0.0,0.0,0.0,1.206154,0.512604,-0.931556,0.0,78.450
1,1.494877,0.0,0.0,0.0,0.0,-1.035098,-1.035098,0.0,-0.784387,0.616770,...,-1.035098,0.0,0.0,0.0,0.0,0.757712,-1.186312,-0.368741,0.0,2.415
2,0.470020,0.0,0.0,0.0,0.0,-1.035098,-1.035098,0.0,1.513334,-0.103177,...,-1.035098,0.0,0.0,0.0,0.0,-1.484497,0.512604,-1.494371,0.0,76.900
3,-0.554836,0.0,0.0,0.0,0.0,-1.035098,-1.035098,0.0,-0.830977,0.831257,...,-1.035098,0.0,0.0,0.0,0.0,-0.139172,0.087875,-0.650148,0.0,2.589
4,-1.579693,0.0,0.0,0.0,0.0,-1.035098,-1.035098,0.0,-0.856996,0.937268,...,-1.035098,0.0,0.0,0.0,0.0,-1.036055,1.362062,-1.775779,0.0,2.675


Model Creation and Evaluation

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler  # though already scaled, kept for safety

# 1. Load the scaled dataset
df = pd.read_csv("./sample_data/Scaled_Common_Records.csv")

# 2. Define features (X) and target (y)
X = df.drop(columns=['price'])
y = df['price']

# 3. Train/Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

# 4. Define a dictionary of models to evaluate
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.1),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'SVR (RBF)': SVR(kernel='rbf', C=1.0, gamma='scale'),
    'KNN': KNeighborsRegressor(n_neighbors=3)
}

# 5. Evaluate each model using cross-validation on training set
print("\n=== Cross-Validation R² (5-fold) on Training Set ===")
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    cv_results[name] = scores.mean()
    print(f"{name:20s} : Mean R² = {scores.mean():.4f} ± {scores.std():.4f}")

# 6. Select the best model based on CV score
best_name = max(cv_results, key=cv_results.get)
best_model = models[best_name]
print(f"\nBest model selected: {best_name}")

# 7. Train the best model on the entire training set
best_model.fit(X_train, y_train)

# 8. Predict on the test set
y_pred = best_model.predict(X_test)

# 9. Compute R² and Adjusted R² on test set
n = X_test.shape[0]          # number of test samples
p = X_test.shape[1]          # number of features
r2 = r2_score(y_test, y_pred)
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print("\n=== Test Set Performance ===")
print(f"R² score          : {r2:.4f}")
print(f"Adjusted R² score : {adj_r2:.4f}")

# 10. (Optional) Evaluate all models on test set for comparison
print("\n=== All Models – Test Set R² ===")
for name, model in models.items():
    model.fit(X_train, y_train)   # refit each
    pred = model.predict(X_test)
    r2_test = r2_score(y_test, pred)
    print(f"{name:20s} : R² = {r2_test:.4f}")

# 11. Feature importance (if applicable)
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feature_names = X.columns
    importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    importance_df = importance_df.sort_values('Importance', ascending=False)
    print("\n=== Feature Importances (Best Model) ===")
    print(importance_df.head(10))

Training samples: 23, Test samples: 6

=== Cross-Validation R² (5-fold) on Training Set ===
Linear Regression    : Mean R² = 0.8758 ± 0.1356
Ridge Regression     : Mean R² = 0.9721 ± 0.0150
Lasso Regression     : Mean R² = 0.6860 ± 0.6104
Random Forest        : Mean R² = 0.8911 ± 0.1467
Gradient Boosting    : Mean R² = 0.9052 ± 0.1895
SVR (RBF)            : Mean R² = -1.0998 ± 0.9501
KNN                  : Mean R² = 0.4905 ± 0.5080

Best model selected: Ridge Regression

=== Test Set Performance ===
R² score          : 0.9755
Adjusted R² score : 1.0027

=== All Models – Test Set R² ===
Linear Regression    : R² = 0.9950
Ridge Regression     : R² = 0.9755
Lasso Regression     : R² = 0.9995
Random Forest        : R² = 0.9715
Gradient Boosting    : R² = 1.0000
SVR (RBF)            : R² = -0.3968
KNN                  : R² = 0.8307


In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()
price_corr = corr_matrix["price"].sort_values(ascending=False)
price_corr

,price
price,1.000000
total_notional_value,0.972168
unit_of_measure_BBL,0.734526
product_grade_le_encoded,0.667727
product_grade_encoded,0.667727
quantity_bbl,0.629548
market_region_North Sea,0.597117
unit_of_measure_Barrel,0.476095
broker_Commerical Bank A,0.329636
broker_International Broker C,0.318247


In [ ]:
plt.figure(figsize=(4, 6))
sns.heatmap(
    price_corr.to_frame(),
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    cbar=True,
)
plt.title("Feature Correlation with Price")
plt.show()

NameError: name 'sns' is not defined

<Figure size 400x600 with 0 Axes>

Cross-Validation (CV) R² on Training:

Linear Regression: 0.8758 ± 0.1356 (Good, but somewhat unstable across folds).

Ridge Regression: 0.9721 ± 0.0150 (Excellent, very stable low variance). Selected as best.

Lasso Regression: 0.6860 ± 0.6104 (Terrible and highly unstable - huge std dev, meaning it performs poorly on some folds, likely due to the small dataset and feature sparsity).

Random Forest: 0.8911 ± 0.1467 (Good, but high variance).

Gradient Boosting: 0.9052 ± 0.1895 (Good, but high variance).

SVR: -1.0998 ± 0.9501 (Horrible, completely fails - R² below 0 means worse than guessing the mean, huge variance).

KNN: 0.4905 ± 0.5080 (Poor, unstable).

Test Set Performance:

Ridge R² = 0.9755 (Excellent - explains 97.55% of the variance in the test set).

Ridge Adjusted R² = 1.0027 (Greater than 1! This is mathematically impossible for true R², but happens due to small sample size n and large number of features p making the denominator tiny, or perhaps overfitting to the specific test split).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score

# ----------------------------------------------------------------------
# 1. LOAD DATA (assuming df, X, y, X_train, y_train, best_model are already defined)
# If running separately, reload the scaled CSV:
# df = pd.read_csv("./sample_data/Scaled_Common_Records.csv")
# X = df.drop(columns=['price'])
# y = df['price']
# X_train, X_test, y_train, y_test = train_test_split(...)  # use the same split
# best_model = Ridge(alpha=1.0)  # or load from previous run
# ----------------------------------------------------------------------

# Set style for all plots
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# -------------------------------------------
# DESCRIPTIVE ANALYSIS VISUALIZATIONS
# -------------------------------------------

# 1.1 Distribution of Target Variable (price)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['price'], kde=True, ax=ax[0])
ax[0].set_title('Price Distribution (Histogram + KDE)')
ax[0].set_xlabel('Price (USD)')

sns.boxplot(x=df['price'], ax=ax[1])
ax[1].set_title('Price Boxplot')
fig.tight_layout()
plt.savefig('descriptive_price_distribution.png', dpi=300)
plt.show()

# 1.2 Correlation Heatmap (top 15 features with highest correlation to price)
# Compute correlations with target
correlations = df.corr()['price'].sort_values(ascending=False)
top_features = correlations.head(15).index.tolist()  # include price itself

# Heatmap of these features
plt.figure(figsize=(12, 10))
corr_matrix = df[top_features].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', square=True, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap (Top 15 Features vs. Price)', fontsize=14)
plt.tight_layout()
plt.savefig('descriptive_correlation_heatmap.png', dpi=300)
plt.show()

# 1.3 Boxplots for key categorical-like features (if any)
# For example, if you have encoded regions, show price by region (use original columns if present)
# Since we have dummy variables, we can group by the original region columns (one-hot)
# Let's create a region label from the one-hot encoded columns (assume they exist)
region_cols = [col for col in df.columns if col.startswith('market_region_')]
if region_cols:
    # For each row, find which region is 1 (if exactly one)
    # We'll use idxmax to get the region name
    df['region'] = df[region_cols].idxmax(axis=1).str.replace('market_region_', '')
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='region', y='price', data=df)
    plt.title('Price Distribution by Market Region')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('descriptive_price_by_region.png', dpi=300)
    plt.show()
    df.drop(columns=['region'], inplace=True)  # clean up

# -------------------------------------------
# MACHINE LEARNING DIAGNOSTIC CHARTS
# -------------------------------------------

# 2.1 Cross-Validation Scores Bar Chart (using the results from earlier)
# We'll re-run CV on all models (or just the best model) to get scores for plotting
# Let's assume we have a dictionary 'models' and 'cv_results' from previous step.
# If not, we can compute for the best model:

cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='r2')
plt.figure(figsize=(8, 5))
plt.bar(range(1, 6), cv_scores, color='skyblue', edgecolor='black')
plt.axhline(y=cv_scores.mean(), color='red', linestyle='--', label=f'Mean R² = {cv_scores.mean():.3f}')
plt.xlabel('Fold')
plt.ylabel('R² Score')
plt.title('5‑Fold Cross‑Validation R² for Ridge Regression')
plt.ylim(0.9, 1.0)  # adjust based on your scores
plt.legend()
plt.tight_layout()
plt.savefig('ml_cv_scores.png', dpi=300)
plt.show()

# 2.2 Actual vs. Predicted Scatter Plot (on test set)
# Assume y_pred already computed
# If not, compute:
# y_pred = best_model.predict(X_test)

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.7, edgecolors='k')
# Add perfect prediction line
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs. Predicted Prices (Test Set)')
plt.legend()
plt.tight_layout()
plt.savefig('ml_actual_vs_predicted.png', dpi=300)
plt.show()

# 2.3 Residuals Plot (Predicted vs. Residuals)
residuals = y_test - y_pred
plt.figure(figsize=(8, 6))
plt.scatter(y_pred, residuals, alpha=0.7, edgecolors='k')
plt.axhline(y=0, color='red', linestyle='--', lw=2)
plt.xlabel('Predicted Price')
plt.ylabel('Residuals (Actual - Predicted)')
plt.title('Residuals vs. Predicted Values')
plt.tight_layout()
plt.savefig('ml_residuals_plot.png', dpi=300)
plt.show()

# 2.4 Feature Importance (Coefficient Magnitude for Ridge)
# Ridge has coefficients; we can plot top coefficients
coefs = best_model.coef_
feature_names = X.columns
coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefs})
coef_df['Abs_Coefficient'] = np.abs(coef_df['Coefficient'])
coef_df = coef_df.sort_values('Abs_Coefficient', ascending=False).head(15)

plt.figure(figsize=(10, 8))
sns.barplot(x='Coefficient', y='Feature', data=coef_df, palette='coolwarm')
plt.axvline(x=0, color='black', linestyle='-', lw=0.5)
plt.title('Top 15 Feature Coefficients (Ridge Regression)')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.savefig('ml_feature_importance.png', dpi=300)
plt.show()

# 2.5 (Optional) Learning Curve to see if more data would help
from sklearn.model_selection import learning_curve

train_sizes, train_scores, test_scores = learning_curve(
    best_model, X_train, y_train, cv=5,
    train_sizes=np.linspace(0.1, 1.0, 5),
    scoring='r2'
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_mean, 'o-', color='blue', label='Training R²')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2, color='blue')
plt.plot(train_sizes, test_mean, 'o-', color='green', label='Cross-Validation R²')
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.2, color='green')
plt.xlabel('Training Examples')
plt.ylabel('R² Score')
plt.title('Learning Curve (Ridge Regression)')
plt.legend(loc='best')
plt.grid(True)
plt.tight_layout()
plt.savefig('ml_learning_curve.png', dpi=300)
plt.show()

print("\nAll charts saved to the current directory.")